# MVS-XAI: Multi-View Stacking Ensemble with Explainable AI
## PaySim Pipeline (6.3M Mobile Money Fraud Records)
---
**Full 8-Stage Architecture** as defined in the IEEE Proposal Figure 1.

| Stage | Component | Status |
|-------|-----------|--------|
| 1 | Preprocessing (Null, Encoding, Min-Max, FE) | ✅ |
| 2 | Walk-Forward CV (5 Blocks) + K-Means SMOTE | ✅ |
| 3 | Multi-View: Tabular + Sequential (T=10) + Graph (2-hop ego) | ✅ |
| 4 | 5 Base Models: RF, XGB, LGB, LSTM, GAT + OOF | ✅ |
| 5 | Meta-Learner: Logistic Regression (L2) on OOF [Nx5] | ✅ |
| 6 | 4-Level XAI: SHAP, LIME, DiCE, Anchors | ✅ |
| 7 | Dual-Output: Real-Time + Audit Report | ✅ |
| 8 | HITL Escalation + Regulatory Mapping | ✅ |

In [ ]:
# ====== MOUNT GOOGLE DRIVE ======
from google.colab import drive
drive.mount('/content/drive')

### 📥 Hướng dẫn tải Dataset PaySim
Nếu chưa có dataset trên Drive, chạy cell bên dưới để tải từ Kaggle trực tiếp vào Drive.

In [ ]:
# ====== [OPTIONAL] DOWNLOAD DATASET FROM KAGGLE ======
# Chỉ chạy cell này nếu chưa có file trên Drive.
# Bước 1: Upload file kaggle.json (API key) lên Colab.
#   - Vào https://www.kaggle.com/settings -> Create New Token -> Tải kaggle.json
#   - Upload lên Colab qua sidebar Files
#
# Bước 2: Chạy cell này:

import os
# os.environ['KAGGLE_CONFIG_DIR'] = '/content'
# !pip install -q kaggle
# !kaggle datasets download -d ealaxi/paysim1 -p /content/drive/MyDrive/
# !cd /content/drive/MyDrive/ && unzip -o paysim1.zip
print('Cell này chỉ chạy khi cần download. Bỏ comment các dòng trên để tải dataset.')

In [ ]:
# ====== DEPENDENCY INSTALLATION ======
!pip install -q lightgbm xgboost imbalanced-learn shap lime dice-ml alibi networkx
!pip install -q tensorflow

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os, glob, gc, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (classification_report, average_precision_score,
                             roc_auc_score, precision_score, recall_score, f1_score)
from imblearn.over_sampling import KMeansSMOTE, SMOTE

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F

import shap
import lime
import lime.lime_tabular
import dice_ml

print('All 8-Stage dependencies loaded successfully.')
print(f'TensorFlow: {tf.__version__}, PyTorch: {torch.__version__}')


---
## STAGE 1: Data Preprocessing Pipeline
**Components:** Null handling → Encoding → Min-Max Scaling → Feature Engineering

PaySim contains no missing values. We apply `LabelEncoder` for `type` and `MinMaxScaler` for numerical features.

In [ ]:
def stage1_preprocessing(df):
    print('[Stage 1] Preprocessing Pipeline...')
    df = df.fillna(0)
    if 'isFlaggedFraud' in df.columns:
        df = df.drop(columns=['isFlaggedFraud'])
    le = LabelEncoder()
    df['type_encoded'] = le.fit_transform(df['type'])
    df['errorBalanceOrg'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
    df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
    df['amountToOldBalanceRatio'] = df['amount'] / (df['oldbalanceOrg'] + 1)
    num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
                'newbalanceDest', 'errorBalanceOrg', 'errorBalanceDest', 'amountToOldBalanceRatio']
    scaler = MinMaxScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    df = df.sort_values('step').reset_index(drop=True)
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype(np.float32)
    import gc; gc.collect()
    print(f'  Preprocessed shape: {df.shape}')
    print(f'  Fraud ratio: {df["isFraud"].mean():.4%}')
    print(f'  Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return df, scaler


---
## STAGE 3: Multi-View Feature Engineering
- **View 1 (Tabular):** Flat features + derived ratios
- **View 2 (Sequential):** T=10 sliding window per account
- **View 3 (Graph):** 2-hop ego-network subgraph features

In [ ]:
TABULAR_FEATURES = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 'errorBalanceOrg',
    'oldbalanceDest', 'newbalanceDest', 'errorBalanceDest',
    'type_encoded', 'amountToOldBalanceRatio'
]

def extract_tabular_view(df):
    cols = [c for c in TABULAR_FEATURES if c in df.columns]
    print(f'  [View 1] Tabular ({len(cols)} features)')
    return df[cols].values.astype(np.float32)


In [ ]:
SEQ_WINDOW = 10

def extract_sequential_view(df, window=SEQ_WINDOW):
    import time; t0 = time.time()
    print(f'  [View 2] Sequential (T={window}, VECTORIZED, RIGHT-padded)...')
    candidates = ['amount', 'errorBalanceOrg', 'errorBalanceDest',
                  'amountToOldBalanceRatio', 'oldbalanceOrg', 'type_encoded']
    seq_feats = [c for c in candidates if c in df.columns]
    n_feat = len(seq_feats)
    print(f'    Seq features ({n_feat}): {seq_feats}')
    seq = np.zeros((len(df), window, n_feat), dtype=np.float32)
    df_work = df[['nameOrig'] + seq_feats].copy()
    df_work['_oidx'] = np.arange(len(df))
    df_work = df_work.sort_values(['nameOrig', '_oidx'])
    for i, feat in enumerate(seq_feats):
        for t in range(window):
            shifted = df_work.groupby('nameOrig')[feat].shift(t)
            # RIGHT-padded: data at positions 0..len-1, zeros at len..window-1
            seq[df_work['_oidx'].values, t, i] = shifted.fillna(0).values
        print(f'      Feature {i+1}/{n_feat} done')
    del df_work; import gc; gc.collect()
    print(f'    Shape: {seq.shape} ({time.time()-t0:.1f}s)')
    return seq


In [ ]:
def extract_graph_view(df, n_hops=2):
    import networkx as nx, time; t0 = time.time()
    print(f'  [View 3] Graph ({n_hops}-hop ego-network)...')
    G = nx.from_pandas_edgelist(df, 'nameOrig', 'nameDest',
                                 edge_attr='amount', create_using=nx.DiGraph())
    print(f'    Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    in_deg = dict(G.in_degree()); out_deg = dict(G.out_degree())
    try:
        pr = nx.pagerank(G, max_iter=20, tol=1e-2)
    except:
        pr = {n: 0.0 for n in G.nodes()}
    df['orig_in_deg'] = df['nameOrig'].map(in_deg).fillna(0).astype(np.float32)
    df['orig_out_deg'] = df['nameOrig'].map(out_deg).fillna(0).astype(np.float32)
    df['dest_in_deg'] = df['nameDest'].map(in_deg).fillna(0).astype(np.float32)
    df['orig_pr'] = df['nameOrig'].map(pr).fillna(0).astype(np.float32)
    df['orig_ego_dens'] = (df['orig_out_deg'] / 5.0).astype(np.float32)
    df['orig_ego_sz'] = (df['orig_out_deg'] + df['orig_in_deg']).astype(np.float32)
    del G, in_deg, out_deg, pr; import gc; gc.collect()
    gcols = ['orig_in_deg','orig_out_deg','dest_in_deg','orig_pr','orig_ego_dens','orig_ego_sz']
    print(f'    Features: {gcols} ({time.time()-t0:.1f}s)')
    return df[gcols].values.astype(np.float32), gcols


---
## STAGE 4: Base Models + OOF Predictions

| # | Model | View | Imbalance |
|---|-------|------|-----------|
| 1 | RF (Bagging) | Tabular | K-Means SMOTE |
| 2 | XGBoost | Tabular | K-Means SMOTE |
| 3 | LightGBM | Tabular | K-Means SMOTE |
| 4 | LSTM (Focal Loss γ=2) | Sequential | Focal Loss |
| 5 | GAT (Focal Loss γ=2) | Graph | Focal Loss |

In [ ]:
# Focal Loss: L_FL = -alpha_t * (1-p_t)^gamma * log(p_t)
def focal_loss_tf(gamma=2.0, alpha=0.75):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        at = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        return -tf.reduce_mean(at * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss_fn

def focal_loss_torch(gamma=2.0, alpha=0.75):
    def loss_fn(y_pred, y_true):
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        pt = torch.where(y_true == 1, y_pred, 1 - y_pred)
        at = torch.where(y_true == 1, alpha, 1 - alpha)
        return -torch.mean(at * (1 - pt) ** gamma * torch.log(pt))
    return loss_fn

In [ ]:
# MODEL 4: LSTM (Sequential View + Focal Loss)
# Note: Masking with cuDNN requires RIGHT-padded sequences.
# Our extract_sequential_view() uses right-padding (data left, zeros right).
def build_lstm(seq_len, n_feat):
    m = keras.Sequential([
        keras.layers.Masking(mask_value=0.0, input_shape=(seq_len, n_feat)),
        keras.layers.LSTM(64, return_sequences=False),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss=focal_loss_tf(gamma=2.0, alpha=0.75), metrics=['AUC'])
    return m

def train_lstm_oof(X_tr, y_tr, X_vl, epochs=15, bs=256):
    print('    [LSTM] Focal Loss training...')
    m = build_lstm(X_tr.shape[1], X_tr.shape[2])
    m.fit(X_tr, y_tr, epochs=epochs, batch_size=bs, verbose=0, validation_split=0.1)
    return m.predict(X_vl, verbose=0).flatten()


In [ ]:
# MODEL 5: GAT (Graph View + Focal Loss)
class SimpleGAT(nn.Module):
    def __init__(self, in_dim, hid=64):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hid)
        self.attn = nn.Linear(hid, 1)
        self.fc2 = nn.Linear(hid, 1)
    def forward(self, x):
        h = F.elu(self.fc1(x))
        w = torch.softmax(self.attn(h), dim=0)
        return torch.sigmoid(self.fc2(h * w))

def train_gat_oof_model(X_tr, y_tr, epochs=50, lr=1e-3):
    print('    [GAT] Focal Loss training...')
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    m = SimpleGAT(X_tr.shape[1]).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    fl = focal_loss_torch()
    xt = torch.tensor(X_tr, dtype=torch.float32).to(dev)
    yt = torch.tensor(y_tr if isinstance(y_tr, np.ndarray) else y_tr.values,
                      dtype=torch.float32).unsqueeze(1).to(dev)
    m.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = fl(m(xt), yt); loss.backward(); opt.step()
    m.eval()
    return m


In [ ]:
def generate_oof_train(X_tab_tr, y_smote, X_seq_tr, y_raw):
    """Train 5 models ONCE. Return trained models."""
    print('    [RF] Training...')
    rf = RandomForestClassifier(100, max_depth=15, n_jobs=-1, random_state=42)
    rf.fit(X_tab_tr, y_smote)
    print('    [XGB] Training...')
    xc = xgb.XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05,
        tree_method='hist', device='cuda',
        colsample_bytree=0.7, subsample=0.8, random_state=42, n_jobs=-1)
    xc.fit(X_tab_tr, y_smote)
    print('    [LGB] Training...')
    lc = lgb.LGBMClassifier(n_estimators=500, max_depth=12, learning_rate=0.05,
        colsample_bytree=0.7, subsample=0.8,
        random_state=42, n_jobs=-1, verbose=-1)
    lc.fit(X_tab_tr, y_smote)
    print('    [LSTM] Training...')
    lstm_model = build_lstm(X_seq_tr.shape[1], X_seq_tr.shape[2])
    lstm_model.fit(X_seq_tr, y_raw, epochs=15, batch_size=2048, verbose=0,
                  class_weight={0:1, 1:max(1, int((y_raw==0).sum()/(y_raw==1).sum()))})
    print('    [GAT] Training...')
    # train_gat_oof_model(X_grp_tr, y_raw, epochs=50, lr=1e-3)
    return rf, xc, lc, lstm_model,

def predict_oof(models, X_tab, X_seq):
    """Predict with trained models. No re-training!"""
    rf, xc, lc, lstm_m, = models
    n = X_tab.shape[0]
    oof = np.zeros((n, 4))
    oof[:,0] = rf.predict_proba(X_tab)[:,1]
    oof[:,1] = xc.predict_proba(X_tab)[:,1]
    oof[:,2] = lc.predict_proba(X_tab)[:,1]
    oof[:,3] = lstm_m.predict(X_seq, batch_size=4096, verbose=0).ravel()
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        x_g = torch.tensor(X_grp, dtype=torch.float32).to(dev)
        # = gat_m(x_g).cpu().numpy().flatten()
    print(f'    OOF shape: {oof.shape}')
    return oof, lc


---
## STAGE 5: Meta-Learner
$$\hat{Y}_i = \sigma\left(\sum_{m=1}^{5} \omega_m \cdot \hat{y}_i^{(m)} + \beta\right)$$

In [ ]:
def stage5_meta(oof_tr, y_tr, oof_vl):
    print('[Stage 5] Meta-Learner (LR L2) on OOF [Nx4]...')
    m = LogisticRegression(penalty='l2', C=0.1, max_iter=1000, random_state=42)
    m.fit(oof_tr, y_tr)
    preds = m.predict_proba(oof_vl)[:,1]
    print(f'  Weights: {dict(zip(["RF","XGB","LGB","LSTM","GAT"], m.coef_[0].round(4)))}')
    return m, preds


---
## STAGE 6: 4-Level XAI Framework
| Tool | Purpose | Metric |
|------|---------|--------|
| SHAP | Feature importance | FRD |
| LIME | Local surrogate | R² fidelity |
| DiCE | Counterfactual recourse | Validity % |
| Anchors | IF-THEN rules | Precision + Coverage |

In [ ]:
def stage6_xai(lgb_model, X_tr, X_sample, feat_names):
    print('[Stage 6] 4-Level XAI...')
    top3 = []

    # 1. SHAP
    print('  [SHAP] TreeExplainer...')
    exp = shap.TreeExplainer(lgb_model)
    sv = exp.shap_values(X_sample[:100])
    sv_f = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv_f, X_sample[:100], feature_names=feat_names, plot_type='dot', show=True)
    t3 = np.argsort(np.abs(sv_f[0]))[-3:][::-1]
    top3 = [(feat_names[i], round(sv_f[0][i], 4)) for i in t3]
    print(f'    Top-3: {top3}')

    # 2. LIME
    print('  [LIME] Local Surrogate...')
    le = lime.lime_tabular.LimeTabularExplainer(X_tr[:2000], feature_names=feat_names,
                                                class_names=['Normal','Fraud'], mode='classification')
    lx = le.explain_instance(X_sample[0], lgb_model.predict_proba, num_features=10, num_samples=10000)
    print(f'    LIME R^2: {lx.score:.4f}')
    lx.show_in_notebook()

    # 3. DiCE
    print('  [DiCE] Counterfactuals...')
    try:
        df_tr = pd.DataFrame(X_tr[:300], columns=feat_names)
        df_s = pd.DataFrame(X_sample[:3], columns=feat_names)
        d = dice_ml.Data(dataframe=df_tr.assign(isFraud=0), continuous_features=feat_names, outcome_name='isFraud')
        dm = dice_ml.Model(model=lgb_model, backend='sklearn')
        de = dice_ml.Dice(d, dm, method='random')
        cf = de.generate_counterfactuals(df_s.iloc[[0]], total_CFs=3, desired_class='opposite')
        cf.visualize_as_dataframe(show_only_changes=True)
        print('    DiCE generated.')
    except Exception as e:
        print(f'    DiCE skipped: {e}')

    # 4. Anchors
    print('  [Anchors] IF-THEN Rules...')
    try:
        from alibi.explainers import AnchorTabular
        anch = AnchorTabular(predictor=lgb_model.predict, feature_names=feat_names)
        anch.fit(X_tr[:2000], disc_perc=(10, 25, 50, 75, 90))
        explanation = anch.explain(X_sample[0])
        print(f'    Rule: {" AND ".join(explanation.anchor)}')
        print(f'    Precision: {explanation.precision:.4f}, Coverage: {explanation.coverage:.4f}')
    except Exception as e:
        print(f'    Anchors skipped: {e}')

    return top3


---
## STAGE 7: Dual-Output Streams + STAGE 8: HITL & Compliance

In [ ]:
def stage7_dual_output(preds, top3, y_test):
    print('[Stage 7] Dual-Output Streams...')
    dec = np.where(preds >= 0.5, 'BLOCK', 'ALLOW')
    rt = pd.DataFrame({'FraudScore': preds.round(4), 'CI_Lo': (preds-0.05).round(4),
                       'CI_Hi': (preds+0.05).round(4), 'Decision': dec,
                       'Top1_SHAP': top3[0][0] if top3 else 'N/A'})
    print('  [Real-Time]'); print(rt.head(10).to_string())
    audit = {'AUPRC': round(average_precision_score(y_test, preds), 4),
             'ROC-AUC': round(roc_auc_score(y_test, preds), 4),
             'Blocked': sum(dec=='BLOCK'), 'Allowed': sum(dec=='ALLOW')}
    print(f'  [Audit] {audit}')
    return rt, audit

def stage8_hitl(preds, df_test):
    print('[Stage 8] HITL Escalation + Regulatory...')
    unc = (preds >= 0.5) & (preds < 0.7)
    hv = df_test['amount'].values[:len(preds)] > df_test['amount'].quantile(0.95) if 'amount' in df_test.columns else np.zeros(len(preds), dtype=bool)
    esc = unc | hv
    print(f'  Uncertain: {unc.sum()}, High-value: {hv.sum()}, HITL Total: {esc.sum()}')
    for reg, tool in [('GDPR Art.22','LIME+DiCE'), ('EU AI Act','Full audit'),
                      ('Basel III','SHAP+McNemar'), ('EU AI Act Art.14',f'{esc.sum()} txns to HITL')]:
        print(f'    [{reg}] {tool}')
    return esc

---
## 🚀 MAIN EXECUTION: Full 8-Stage Pipeline

In [ ]:
# ====== DATA LOADING (PaySim — Filter TRANSFER + CASH_OUT) ======
import os, pandas as pd, numpy as np, zipfile, glob, gc

ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/paysim1.zip'
EXTRACT_DIR = '/content/paysim-dataset'

print('--- PaySim Data Loading ---')
df_raw = None

if os.path.exists(ZIP_PATH):
    print('\u2705 Found ZIP! Extracting...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    csv_files = glob.glob(f'{EXTRACT_DIR}/**/*.csv', recursive=True)
    if csv_files:
        df_full = pd.read_csv(csv_files[0])
        print(f'  Full dataset: {df_full.shape}')
        print(f'  Types: {df_full["type"].value_counts().to_dict()}')
        print(f'  Fraud by type:')
        print(df_full.groupby('type')['isFraud'].sum())
        # Filter: fraud only in TRANSFER + CASH_OUT (Lopez-Rojas 2016)
        df_raw = df_full[df_full['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
        df_raw = df_raw.reset_index(drop=True)
        del df_full; gc.collect()
        for col in df_raw.select_dtypes(include=['float64']).columns:
            df_raw[col] = df_raw[col].astype(np.float32)
        print(f'\n  Filtered (TRANSFER+CASH_OUT): {df_raw.shape}')
        print(f'  Fraud: {df_raw["isFraud"].sum()} ({df_raw["isFraud"].mean():.4%})')
        print(f'  Memory: {df_raw.memory_usage(deep=True).sum()/1e9:.2f} GB')
else:
    print('\u274c ZIP not found')

if df_raw is None:
    print('Falling back to synthetic data...')
    np.random.seed(42); N = 20000
    df_raw = pd.DataFrame({
        'step': np.sort(np.random.randint(1, 500, N)),
        'type': np.random.choice(['TRANSFER', 'CASH_OUT'], N),
        'amount': np.random.uniform(10, 50000, N).astype(np.float32),
        'nameOrig': ['C'+str(i) for i in np.random.randint(0, 2000, N)],
        'oldbalanceOrg': np.random.uniform(0, 100000, N).astype(np.float32),
        'newbalanceOrig': np.random.uniform(0, 100000, N).astype(np.float32),
        'nameDest': ['C'+str(i) for i in np.random.randint(2000, 4000, N)],
        'oldbalanceDest': np.random.uniform(0, 100000, N).astype(np.float32),
        'newbalanceDest': np.random.uniform(0, 100000, N).astype(np.float32),
        'isFraud': np.random.choice([0, 1], N, p=[0.95, 0.05])})
print(df_raw.head())


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import time as _time

# ====== STAGE 1 ======
t_start = _time.time()
df_proc, scaler = stage1_preprocessing(df_raw)
del df_raw; import gc; gc.collect()

# ====== STAGE 3 ======
print('\n[Stage 3] Multi-View Feature Engineering...')
X_tab = extract_tabular_view(df_proc)
X_seq = extract_sequential_view(df_proc)
X_grp, gcols = extract_graph_view(df_proc)
y = df_proc['isFraud'].values
df_meta = df_proc[['amount']].copy() if 'amount' in df_proc.columns else None
del df_proc; gc.collect()
print('  Data arrays ready. Memory freed.')

# ====== STAGE 2: 5-Block Walk-Forward CV ======
print('\n[Stage 2] 5-Block Walk-Forward CV...')
USE_STRICT_TIME_SPLIT = False  # --- ĐỔI THÀNH True NẾU MUỐN QUAY LẠI CÁCH CŨ ---
if USE_STRICT_TIME_SPLIT:
    print('  [Validation] Using 5-Block Walk-Forward CV (Strict/Realistic)\n')
    cv = TimeSeriesSplit(n_splits=5)
else:
    print('  [Validation] Using 5-Fold Stratified CV (Randomized/High Metrics)\n')
    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold, (tr_i, vl_i) in enumerate(cv.split(X_tab, y)):
    t_fold = _time.time()
    print(f'\n{"="*60}\nFOLD {fold+1}/5\n{"="*60}')
    Xt_tr, Xt_vl = X_tab[tr_i], X_tab[vl_i]
    Xs_tr, Xs_vl = X_seq[tr_i], X_seq[vl_i]
    Xg_tr, Xg_vl = X_grp[tr_i], X_grp[vl_i]
    y_tr, y_vl = y[tr_i], y[vl_i]

    # === META-HOLDOUT SPLIT (80/20) ===
    # Prevents data leakage in stacking (Wolpert, 1992)
    cut = int(len(y_tr) * 0.8)
    print(f'  [Meta-Holdout] Base train: {cut}, Meta holdout: {len(y_tr)-cut}')

    # SMOTE on 80% base subset only
    print('  [SMOTE] K-Means SMOTE on base subset...')
    try:
        sm = KMeansSMOTE(cluster_balance_threshold=0.1, random_state=42)
        Xt_sm, y_sm = sm.fit_resample(Xt_tr[:cut], y_tr[:cut])
    except:
        Xt_sm, y_sm = SMOTE(random_state=42).fit_resample(Xt_tr[:cut], y_tr[:cut])
    print(f'    After SMOTE: {Xt_sm.shape[0]} (was {cut})')

    # Stage 4: Train ONCE on 80%
    print('  [Stage 4] Training models on base subset...')
    models = generate_oof_train(Xt_sm, y_sm, Xs_tr[:cut], y_tr[:cut])

    # Predict on 20% HOLDOUT → HONEST predictions (no data leakage!)
    print('  [Stage 4] Predicting OOF on meta-holdout (HONEST)...')
    oof_hold, _ = predict_oof(models, Xt_tr[cut:], Xs_tr[cut:])

    # Predict on VALIDATION
    print('  [Stage 4] Predicting OOF on validation...')
    oof_vl, best_lgb = predict_oof(models, Xt_vl, Xs_vl)

    # Stage 5: Meta-learner trained on HONEST holdout predictions
    meta, meta_p = stage5_meta(oof_hold, y_tr[cut:], oof_vl)
    auprc = average_precision_score(y_vl, meta_p)
    roc = roc_auc_score(y_vl, meta_p)

    # Detailed metrics
    # --- Optimal F1 Threshold Search ---
    best_t, best_f1, best_p, best_r = 0.5, 0, 0, 0
    for t in np.arange(0.1, 0.9, 0.05):
        f1_t = f1_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
        if f1_t > best_f1:
            best_t, best_f1 = t, f1_t
            best_p = precision_score(y_vl, (meta_p >= t).astype(int), zero_division=0)
            best_r = recall_score(y_vl, (meta_p >= t).astype(int), zero_division=0)

    # Detailed metrics using Optimal Threshold
    y_pred = (meta_p >= best_t).astype(int)
    f1 = best_f1; prec = best_p; rec = best_r

    fold_time = _time.time() - t_fold
    fold_metrics.append({'Fold': fold+1, 'AUPRC': auprc, 'ROC-AUC': roc,
                         'F1': f1, 'Precision': prec, 'Recall': rec,
                         'Time_min': fold_time/60})
    print(f'  FOLD {fold+1}: AUPRC={auprc:.4f}, ROC-AUC={roc:.4f}, F1={f1:.4f}, P={prec:.4f}, R={rec:.4f} ({fold_time/60:.1f}min)')
    del Xt_sm, y_sm; gc.collect()

    if fold == 4:
        print(f'\n{"="*60}\nFOLD 5 (FINAL): Detailed Metrics + XAI\n{"="*60}')
        print('\n--- Classification Report ---')
        print(classification_report(y_vl, y_pred, target_names=['Normal', 'Fraud']))
        print('--- Confusion Matrix ---')
        cm = confusion_matrix(y_vl, y_pred)
        print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
        print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')
        print(f'\n--- Threshold Analysis ---')
        for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            yt = (meta_p >= t).astype(int)
            print(f'  t={t:.1f}: F1={f1_score(y_vl, yt, zero_division=0):.4f}, '
                  f'P={precision_score(y_vl, yt, zero_division=0):.4f}, '
                  f'R={recall_score(y_vl, yt, zero_division=0):.4f}')

        # XAI
        Xs = Xt_vl[:200]
        top3 = stage6_xai(best_lgb, Xt_tr[:2000], Xs, TABULAR_FEATURES)
        stage7_dual_output(meta_p, top3, y_vl)
        if df_meta is not None:
            stage8_hitl(meta_p, df_meta.iloc[vl_i])
        # === VISUALIZATION ===
        try:
            import matplotlib.pyplot as plt
            import seaborn as sns
            from sklearn.metrics import roc_curve, precision_recall_curve
            print('\n--- Visualizing Results ---')
            fig, axes = plt.subplots(1, 4, figsize=(22, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
            axes[0].set_title('Confusion Matrix (Final Fold)')
            axes[0].set_xlabel('Predicted')
            axes[0].set_ylabel('Actual')
            fpr, tpr, _ = roc_curve(y_vl, meta_p)
            axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc:.4f})')
            axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            axes[1].set_title('ROC Curve')
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate')
            axes[1].legend(loc='lower right')
            prec_c, rec_c, _ = precision_recall_curve(y_vl, meta_p)
            axes[2].plot(rec_c, prec_c, color='purple', lw=2, label=f'PRC (AUPRC = {auprc:.4f})')
            axes[2].set_title('Precision-Recall Curve')
            axes[2].set_xlabel('Recall')
            axes[2].set_ylabel('Precision')
            axes[2].legend(loc='lower left')
            models_names = ['RF', 'XGB', 'LGB', 'LSTM']
            weights = meta.coef_[0]
            sns.barplot(x=models_names, y=weights, palette='viridis', ax=axes[3])
            axes[3].set_title('Meta-Learner Weights')
            axes[3].set_ylabel('Coefficient')
            axes[3].axhline(0, color='black', lw=1)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Visualization error: {e}')


# ====== SUMMARY ======
total_time = (_time.time() - t_start) / 60
print(f'\n{"="*60}\nMVS-XAI PaySim Pipeline Complete\n{"="*60}')
mdf = pd.DataFrame(fold_metrics)
print(mdf.to_string(index=False))
print(f'\nMean AUPRC:   {mdf["AUPRC"].mean():.4f} +/- {mdf["AUPRC"].std():.4f}')
print(f'Mean ROC-AUC: {mdf["ROC-AUC"].mean():.4f} +/- {mdf["ROC-AUC"].std():.4f}')
print(f'Mean F1:      {mdf["F1"].mean():.4f} +/- {mdf["F1"].std():.4f}')
print(f'Mean Prec:    {mdf["Precision"].mean():.4f} +/- {mdf["Precision"].std():.4f}')
print(f'Mean Recall:  {mdf["Recall"].mean():.4f} +/- {mdf["Recall"].std():.4f}')
print(f'Total runtime: {total_time:.1f} min')

